In [ ]:
from datasets import Dataset

# Convert pandas DataFrames to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(validation_data)

In [ ]:
from transformers import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("t5-small")

def preprocess_function(examples):

    # T5 expects the task prefix
    inputs = ["summarize: " + dialogue for dialogue in examples["dialogue"]]

    # Tokenize dialogues
    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    # Tokenize summaries
    labels = tokenizer(
        text_target=examples["summary"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
train_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

val_dataset = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names
)

In [ ]:
print(train_dataset[0])

In [ ]:
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained("t5-small")

In [ ]:
import torch
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=3,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=5e-5,

    weight_decay=0.01,

    logging_dir="./logs",
    logging_steps=50,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    fp16=torch.cuda.is_available(),

    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

In [ ]:
trainer.train()

In [ ]:
SAVE_PATH = "./t5_dialogue_summarizer"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved successfully!")

In [ ]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

MODEL_PATH = "./t5_dialogue_summarizer"

tokenizer = T5Tokenizer.from_pretrained(MODEL_PATH)
model = T5ForConditionalGeneration.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

In [ ]:
def summarize_dialogue(dialogue):

    # T5 requires the task prefix
    text = "summarize: " + dialogue

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=2
        )

    summary = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return summary

In [ ]:
sample_dialogue = """
Amanda: Hi Tom! Are you free this weekend?
Tom: Yes, I am.
Amanda: Let's go hiking.
Tom: Sounds great! What time?
Amanda: Around 8 AM.
Tom: Perfect. I'll bring snacks.
Amanda: I'll bring water and fruits.
"""

print(summarize_dialogue(sample_dialogue))

In [ ]:
from evaluate import load

rouge = load("rouge")

predictions = []
references = []

for sample in val_dataset.select(range(100)):
    pred = summarize_dialogue(
        tokenizer.decode(sample["input_ids"], skip_special_tokens=True)
    )

    ref = tokenizer.decode(sample["labels"], skip_special_tokens=True)

    predictions.append(pred)
    references.append(ref)

scores = rouge.compute(
    predictions=predictions,
    references=references
)

print(scores)